In [ ]:
 Load and Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
xl            = pd.read_excel(r"C:\Users\Tanisha Iyer\Desktop\Projects\DSM Project\ocds_mapped_data_fiscal_year_2016_2022_v3.xlsx", sheet_name=None)
main          = xl['main']
awards        = xl['awards']
awards_sup    = xl['awards_suppliers']

TARGET_YEARS  = ['2020-2021', '2021-2022', '2022-2023']
main_23       = main[main['tender_fiscalYear'].isin(TARGET_YEARS)].copy()

# ── Sector map (expand as needed) ────────────────────────────────────────────
SECTOR_MAP = {
    'Public Works Roads Department':              'roads',
    'Public Works Building and NH Department':    'buildings',
    'National Health Mission':                    'health',
    'Department of Water Resources':              'water_sanitation',
    'Assam Power Distribution Company Ltd':       'electricity',
    'Guwahati Municipal Corporation':             'buildings',
    'Bodoland Territorial Council':               'buildings',
    'Bodoland Territorial Council-PWD':           'roads',
    'Industries and Commerce Department':         'other',
    'Home B':                                     'other',
}
main_23['sector'] = main_23['buyer_name'].map(SECTOR_MAP).fillna('other')

# ── Build awards_full: awards joined to tender metadata ──────────────────────
awards_full = (
    awards
    .merge(awards_sup[['_link_awards','name']],
           left_on='_link', right_on='_link_awards')
    .merge(main_23[['_link','buyer_name','sector',
                    'tender_fiscalYear','tender_value_amount',
                    'tender_procurementMethod','tender_numberOfTenderers']],
           left_on='_link_main', right_on='_link')
    .rename(columns={'name':'supplier_name','value_amount':'award_value'})
)

# ── Clean: drop zero/placeholder values ──────────────────────────────────────
awards_clean = awards_full[awards_full['award_value'] > 1].copy()

# ── Price deviation ───────────────────────────────────────────────────────────
awards_clean['price_dev'] = (
    (awards_clean['award_value'] - awards_clean['tender_value_amount'])
    / awards_clean['tender_value_amount']
)

# Winsorize at ±200%
awards_clean['price_dev_w'] = awards_clean['price_dev'].clip(-2.0, 2.0)

print(f"Clean awards: {len(awards_clean):,}")
print(f"Price deviation available: {awards_clean['price_dev_w'].notna().sum():,}")

In [ ]:
Indicator 1 — Price Deviation

In [ ]:
# ── Sector medians ────────────────────────────────────────────────────────────
sector_median = (
    awards_clean.groupby('sector')['price_dev_w']
    .median()
    .rename('sector_median_dev')
)

# ── Buyer × sector median deviation ──────────────────────────────────────────
ind1 = (
    awards_clean
    .groupby(['buyer_name','sector'])['price_dev_w']
    .agg(median_dev='median', count='count')
    .reset_index()
    .merge(sector_median, on='sector')
)

# Flag: buyer median is ABOVE sector median (less discount = more risk)
# "closer to zero or positive" means higher than sector median
ind1['dev_excess']   = ind1['median_dev'] - ind1['sector_median_dev']
ind1['flag_ind1']    = (ind1['dev_excess'] > 0.05).astype(int)  # 5pp above sector median

print("\n── Indicator 1: Price Deviation (flagged buyer-sectors) ──")
print(ind1[ind1['flag_ind1']==1]
      .sort_values('dev_excess', ascending=False)
      [['buyer_name','sector','median_dev','sector_median_dev','dev_excess']]
      .head(15).to_string(index=False))

# ── Visual: box plot of price deviation by sector ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
order = awards_clean.groupby('sector')['price_dev_w'].median().sort_values().index
awards_clean.boxplot(column='price_dev_w', by='sector', ax=ax,
                     boxprops=dict(color='#2E75B6'),
                     medianprops=dict(color='red', linewidth=2))
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_title('Price Deviation by Sector  |  Assam FY 2020–23')
ax.set_xlabel('Sector')
ax.set_ylabel('(Award − Estimate) / Estimate  [winsorized ±200%]')
plt.suptitle('')
plt.tight_layout()
plt.savefig('ind1_price_deviation.png', dpi=150)
plt.show()

In [ ]:
Indicator 2 — Single-Bidder Rate

In [ ]:
# Exclude Single Source method — those are expected to have one bidder
non_single_source = main_23[
    main_23['tender_procurementMethod'] != 'Single'
].copy()

# Count tenders where numberOfTenderers == 1
sbr = (
    non_single_source[non_single_source['tender_numberOfTenderers'].notna()]
    .groupby(['buyer_name','sector'])
    .apply(lambda g: pd.Series({
        'total_tenders':        len(g),
        'single_bidder_count':  (g['tender_numberOfTenderers'] == 1).sum(),
        'single_bidder_rate':   (g['tender_numberOfTenderers'] == 1).mean()
    }))
    .reset_index()
)

# Overall median SBR
overall_median_sbr = sbr['single_bidder_rate'].median()

# Flag: buyer SBR > overall median + 10pp
sbr['flag_ind2'] = (
    sbr['single_bidder_rate'] > overall_median_sbr + 0.10
).astype(int)

ind2 = sbr.copy()

print(f"\n── Indicator 2: Single-Bidder Rate (overall median = {overall_median_sbr:.1%}) ──")
print(ind2[ind2['flag_ind2']==1]
      .sort_values('single_bidder_rate', ascending=False)
      [['buyer_name','sector','total_tenders','single_bidder_rate']]
      .head(15).to_string(index=False))

# ── Visual: bar chart of SBR by buyer (top 20) ───────────────────────────────
top20_sbr = ind2.nlargest(20, 'single_bidder_rate')

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#E74C3C' if f else '#2E75B6' for f in top20_sbr['flag_ind2']]
ax.barh(top20_sbr['buyer_name'], top20_sbr['single_bidder_rate'] * 100, color=colors)
ax.axvline(overall_median_sbr * 100, color='black', linestyle='--',
           label=f'Overall median ({overall_median_sbr:.1%})')
ax.set_xlabel('Single-Bidder Rate (%)')
ax.set_title('Single-Bidder Rate by Buyer  |  Red = Flagged')
ax.legend()
plt.tight_layout()
plt.savefig('ind2_single_bidder.png', dpi=150)
plt.show()

In [ ]:
Indicator 3 — Procurement Method Composition

In [ ]:
# Non-open methods = Limited, Single, Open Limited
NON_OPEN = ['Limited', 'Single', 'Open Limited']

method_comp = (
    main_23
    .groupby(['buyer_name','sector'])
    .apply(lambda g: pd.Series({
        'total_tenders':     len(g),
        'non_open_count':    g['tender_procurementMethod'].isin(NON_OPEN).sum(),
        'non_open_rate':     g['tender_procurementMethod'].isin(NON_OPEN).mean()
    }))
    .reset_index()
)

# Overall median non-open rate
overall_median_method = method_comp['non_open_rate'].median()

# Flag: non-open rate > median + 10pp AND at least 5 tenders
method_comp['flag_ind3'] = (
    (method_comp['non_open_rate'] > overall_median_method + 0.10) &
    (method_comp['total_tenders'] >= 5)
).astype(int)

ind3 = method_comp.copy()

print(f"\n── Indicator 3: Non-Open Method Rate (median = {overall_median_method:.1%}) ──")
print(ind3[ind3['flag_ind3']==1]
      .sort_values('non_open_rate', ascending=False)
      [['buyer_name','sector','total_tenders','non_open_count','non_open_rate']]
      .head(15).to_string(index=False))

# ── Visual: stacked bar of method mix for top buyers ─────────────────────────
top_buyers = main_23['buyer_name'].value_counts().head(15).index
method_mix = (
    main_23[main_23['buyer_name'].isin(top_buyers)]
    .groupby(['buyer_name','tender_procurementMethod'])
    .size()
    .unstack(fill_value=0)
)
method_mix_pct = method_mix.div(method_mix.sum(axis=1), axis=0) * 100

method_mix_pct.plot(kind='barh', stacked=True, figsize=(12,7),
                    colormap='tab10')
plt.xlabel('Share of Tenders (%)')
plt.title('Procurement Method Mix — Top 15 Buyers  |  Assam FY 2020–23')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('ind3_method_mix.png', dpi=150)
plt.show()

In [ ]:
Indicator 4 — Threshold Bunching

In [ ]:
# Thresholds in INR
THRESHOLDS = {
    '₹25 Lakh':  25_00_000,
    '₹1 Crore':  1_00_00_000,
    '₹10 Crore': 10_00_00_000,
}

values = main_23['tender_value_amount'].dropna()
values = values[values > 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bunching_results = {}

for ax, (label, threshold) in zip(axes, THRESHOLDS.items()):
    # Window: 50% below to 50% above threshold
    lo = threshold * 0.50
    hi = threshold * 1.50
    window = values[(values >= lo) & (values <= hi)]

    # Fine histogram bins
    bins = np.linspace(lo, hi, 60)
    counts, edges = np.histogram(window, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2

    # Plot histogram
    ax.bar(centers, counts, width=(bins[1]-bins[0]),
           color='#2E75B6', alpha=0.7, label='Observed')

    # Counterfactual: fit polynomial to bins EXCLUDING the window
    # just around the threshold (±10%)
    excl_lo = threshold * 0.90
    excl_hi = threshold * 1.10
    mask = (centers < excl_lo) | (centers > excl_hi)
    if mask.sum() > 5:
        poly = np.polyfit(centers[mask], counts[mask], deg=3)
        counterfactual = np.polyval(poly, centers)
        ax.plot(centers, counterfactual, 'r--',
                linewidth=2, label='Counterfactual (poly fit)')

    # Threshold line
    ax.axvline(threshold, color='black', linewidth=2,
               linestyle='--', label=f'Threshold ({label})')

    # Bunching mass: observed vs counterfactual just below threshold
    below_mask = (centers >= threshold * 0.85) & (centers < threshold)
    obs_mass   = counts[below_mask].sum()
    cf_mass    = counterfactual[below_mask].sum() if mask.sum() > 5 else np.nan
    excess     = obs_mass - cf_mass if not np.isnan(cf_mass) else np.nan

    bunching_results[label] = {
        'threshold':       threshold,
        'observed_below':  obs_mass,
        'counterfactual':  round(cf_mass, 1) if not np.isnan(cf_mass) else None,
        'excess_mass':     round(excess, 1) if not np.isnan(excess) else None,
    }

    ax.set_title(f'Bunching around {label}')
    ax.set_xlabel('Tender Value (INR)')
    ax.set_ylabel('Number of Tenders')
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'₹{x/1e7:.1f}Cr' if x>=1e7
                              else f'₹{x/1e5:.0f}L'))
    ax.legend(fontsize=8)

plt.suptitle('Threshold Bunching Analysis  |  Assam FY 2020–23', fontsize=13)
plt.tight_layout()
plt.savefig('ind4_bunching.png', dpi=150)
plt.show()

print("\n── Indicator 4: Bunching Results ──")
for label, res in bunching_results.items():
    print(f"\n{label}:")
    for k, v in res.items():
        print(f"  {k}: {v}")

# Indicator 4 score per buyer: % of tenders just below each threshold
def bunching_score(group):
    vals = group['tender_value_amount'].dropna()
    flags = 0
    for thr in THRESHOLDS.values():
        just_below = ((vals >= thr * 0.85) & (vals < thr)).sum()
        flags += just_below
    return flags / len(vals) if len(vals) > 0 else 0

ind4 = (
    main_23.groupby(['buyer_name','sector'])
    .apply(bunching_score)
    .reset_index()
    .rename(columns={0: 'bunching_score'})
)
ind4_median = ind4['bunching_score'].median()
ind4['flag_ind4'] = (ind4['bunching_score'] > ind4_median + 0.05).astype(int)

In [ ]:
Indicator 5 — Supplier-Buyer Stickiness

In [ ]:
def stickiness(group):
    total = group['award_value'].sum()
    if total == 0:
        return np.nan
    top3 = group.groupby('supplier_name')['award_value'].sum().nlargest(3).sum()
    return top3 / total

ind5 = (
    awards_clean
    .groupby(['buyer_name','sector'])
    .apply(stickiness)
    .reset_index()
    .rename(columns={0: 'stickiness_score'})
)

# Flag: stickiness > 75%
ind5['flag_ind5'] = (ind5['stickiness_score'] > 0.75).astype(int)

print("\n── Indicator 5: Supplier-Buyer Stickiness (flagged > 75%) ──")
print(ind5[ind5['flag_ind5']==1]
      .sort_values('stickiness_score', ascending=False)
      [['buyer_name','sector','stickiness_score']]
      .head(15).to_string(index=False))

In [ ]:
Composite Risk Score

In [ ]:
from scipy.stats import percentileofscore

# ── Merge all 5 indicators on buyer × sector ─────────────────────────────────
composite = (
    ind1[['buyer_name','sector','median_dev','flag_ind1']]
    .merge(ind2[['buyer_name','sector','single_bidder_rate','flag_ind2']],
           on=['buyer_name','sector'], how='outer')
    .merge(ind3[['buyer_name','sector','non_open_rate','flag_ind3']],
           on=['buyer_name','sector'], how='outer')
    .merge(ind4[['buyer_name','sector','bunching_score','flag_ind4']],
           on=['buyer_name','sector'], how='outer')
    .merge(ind5[['buyer_name','sector','stickiness_score','flag_ind5']],
           on=['buyer_name','sector'], how='outer')
)

# ── Percentile-rank each indicator (higher rank = more risk) ─────────────────
def pct_rank(series):
    filled = series.fillna(series.median())
    return filled.rank(pct=True)

# Note: for price deviation, HIGHER median_dev = more risk
composite['pct_ind1'] = pct_rank(composite['median_dev'])
composite['pct_ind2'] = pct_rank(composite['single_bidder_rate'])
composite['pct_ind3'] = pct_rank(composite['non_open_rate'])
composite['pct_ind4'] = pct_rank(composite['bunching_score'])
composite['pct_ind5'] = pct_rank(composite['stickiness_score'])

# ── Equal-weight composite ────────────────────────────────────────────────────
IND_COLS = ['pct_ind1','pct_ind2','pct_ind3','pct_ind4','pct_ind5']
composite['risk_score'] = composite[IND_COLS].mean(axis=1)
composite['flags_total'] = (
    composite[['flag_ind1','flag_ind2','flag_ind3','flag_ind4','flag_ind5']]
    .sum(axis=1)
)

# ── Top 20 highest risk buyer-sector pairs ───────────────────────────────────
top_risk = (
    composite
    .sort_values('risk_score', ascending=False)
    .head(20)
    [['buyer_name','sector','risk_score','flags_total',
      'pct_ind1','pct_ind2','pct_ind3','pct_ind4','pct_ind5']]
)
print("\n── Composite Risk Score: Top 20 Buyer-Sector Pairs ──")
print(top_risk.to_string(index=False))

# ── Sensitivity analysis: alternative weightings ─────────────────────────────
weight_schemes = {
    'Equal (baseline)':     [0.20, 0.20, 0.20, 0.20, 0.20],
    'Price + Stickiness':   [0.30, 0.10, 0.10, 0.10, 0.40],
    'Competition focus':    [0.15, 0.40, 0.30, 0.10, 0.05],
    'Bunching focus':       [0.10, 0.20, 0.20, 0.40, 0.10],
}

for scheme_name, weights in weight_schemes.items():
    composite[f'score_{scheme_name}'] = (
        composite[IND_COLS].fillna(0.5)
        .mul(weights).sum(axis=1)
    )

sensitivity = composite[
    ['buyer_name','sector'] +
    [f'score_{s}' for s in weight_schemes]
].sort_values('score_Equal (baseline)', ascending=False).head(20)

print("\n── Sensitivity Analysis: Top 20 by Equal Weight ──")
print(sensitivity.round(3).to_string(index=False))

# ── Check rank stability across schemes 
rank_cols = {}
for scheme in weight_schemes:
    col = f'score_{scheme}'
    rank_cols[f'rank_{scheme}'] = composite[col].rank(ascending=False)

rank_df = pd.DataFrame(rank_cols)
rank_df['rank_range'] = rank_df.max(axis=1) - rank_df.min(axis=1)
composite['rank_stability'] = rank_df['rank_range'].values

print("\n── Most rank-stable high-risk pairs (robust findings) ──")
stable_risk = (
    composite[composite['risk_score'] > 0.70]
    .sort_values('rank_stability')
    .head(10)
    [['buyer_name','sector','risk_score','flags_total','rank_stability']]
)
print(stable_risk.to_string(index=False))